# 📖 AI Document & Book Scanner Engine (Deep Learning Powered)
### MobileNetV3 4-Corner Neural Keypoint Detector, Perspective Homography, Finger Inpainting & PDF Export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

> **Environment Note:** Set your Colab runtime to GPU: **Runtime ➔ Change runtime type ➔ T4 GPU**.

In [ ]:
# @title 🔄 Git Sync: Pull Latest Updates
# Run this cell anytime updates or improvements are pushed to GitHub!
!git pull 2>/dev/null || echo "Working directory is ready."


In [ ]:
# @title 🚀 Step 1: Environment Setup & Dependency Installation
import os
import sys
import subprocess

print("Checking GPU acceleration...")
try:
    import torch
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠️ Running on CPU. (Runtime -> Change runtime type -> T4 GPU recommended)")
except ImportError:
    pass

print("\nInstalling core vision, deep learning ONNX runtime, and PDF compilation libraries...")
!apt-get update -qq > /dev/null
!apt-get install -y -qq tesseract-ocr libtesseract-dev > /dev/null
!pip install -q opencv-python-headless img2pdf pytesseract onnxruntime gradio

print("\n✅ All dependencies successfully installed!")

## 📥 Step 2: Download Pre-Trained Deep Learning Corner Model
Downloads the specialized MobileNetV3 neural network (13.7 MB) trained specifically to locate document corners on desks, floors, and cluttered backgrounds without cutting text.

In [ ]:
# @title 📂 Step 2: Download AI Model & Prepare Directories
import os
import urllib.request

os.makedirs("models", exist_ok=True)
os.makedirs("user_test_images", exist_ok=True)
os.makedirs("pipeline_outputs", exist_ok=True)

MODEL_URL = "https://huggingface.co/spaces/KennethTM/document_corner_detector/resolve/main/models/timm-mobilenetv3_small_100.onnx"
MODEL_PATH = "models/timm-mobilenetv3_small_100.onnx"

if not os.path.exists(MODEL_PATH) or os.path.getsize(MODEL_PATH) < 1000000:
    print("Downloading Deep Learning Corner Detector (13.7 MB)...", end=" ")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Done!")
else:
    print(f"AI Model verified at {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1024/1024:.2f} MB)")


## ⚙️ Step 3: Deep Learning Pipeline Engine
1. **Neural 4-Corner Keypoint Regressor:** Predicts exact outer page corners ignoring internal lines
2. **Perspective Homography Warp:** Mathematical projection mapping tilted trapezoids into orthogonal flat rectangles
3. **Finger / Thumb Inpainting:** Fast margin texture synthesis
4. **vFlat Magic Color Paper Whitening:** Cleans muddy paper background to crisp white while keeping ink strokes vivid

In [ ]:
# @title 🧠 Deep Learning Modules: Corner Keypoint Regressor & Homography Warp
import cv2
import numpy as np
import pytesseract
import time
import onnxruntime as ort

class AIDocumentCornerDetector:
    _session = None

    @classmethod
    def get_session(cls):
        if cls._session is None:
            cls._session = ort.InferenceSession("models/timm-mobilenetv3_small_100.onnx")
        return cls._session

    @staticmethod
    def normalize_image(image, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
        image = (image / 255.0).astype("float32")
        image[:, :, 0] = (image[:, :, 0] - mean[0]) / std[0]
        image[:, :, 1] = (image[:, :, 1] - mean[1]) / std[1]
        image[:, :, 2] = (image[:, :, 2] - mean[2]) / std[2]
        return image

    @staticmethod
    def resize_longest_max_size(image, max_size=224):
        height, width = image.shape[:2]
        ratio = max_size / max(width, height)
        new_width = int(width * ratio)
        new_height = int(height * ratio)
        return cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_LINEAR)

    @staticmethod
    def pad_if_needed(image, target_size=224):
        height, width, _ = image.shape
        y0 = abs((height - target_size) // 2)
        x0 = abs((width - target_size) // 2)
        background = np.zeros((target_size, target_size, 3), dtype="uint8")
        background[y0:(y0 + height), x0:(x0 + width), :] = image
        return background

    @staticmethod
    def heatmap2keypoints(heatmap: np.ndarray, img_size: int = 224) -> list:
        indx = heatmap.reshape(-1, img_size * img_size).argmax(axis=1)
        row = indx // img_size
        col = indx % img_size
        return np.stack((col, row), axis=1).tolist()

    @staticmethod
    def centercrop_keypoints(keypoints, crop_height, crop_width, img_size=224):
        y_diff = (img_size - crop_height) // 2
        x_diff = (img_size - crop_width) // 2
        return [[x - x_diff, y - y_diff] for x, y in keypoints]

    @staticmethod
    def resize_keypoints(keypoints, current_height, current_width, target_height, target_width):
        return [[int((x / current_width) * target_width), int((y / current_height) * target_height)] for x, y in keypoints]

    @classmethod
    def predict_corners(cls, image_bgr: np.ndarray) -> np.ndarray:
        session = cls.get_session()
        input_name = session.get_inputs()[0].name
        output_name = session.get_outputs()[0].name

        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        h, w, _ = image_rgb.shape

        image_resize = cls.resize_longest_max_size(image_rgb)
        h_small, w_small, _ = image_resize.shape
        image_pad = cls.pad_if_needed(image_resize, target_size=224)
        image_norm = cls.normalize_image(image_pad)
        image_array = np.transpose(image_norm, (2, 0, 1))
        image_array = np.expand_dims(image_array, axis=0)

        output = session.run([output_name], {input_name: image_array})
        output_keypoints = cls.heatmap2keypoints(output[0].squeeze())
        crop_keypoints = cls.centercrop_keypoints(output_keypoints, h_small, w_small, 224)
        large_keypoints = cls.resize_keypoints(crop_keypoints, h_small, w_small, h, w)
        return np.float32(large_keypoints)

    @classmethod
    def warp_perspective(cls, image_bgr: np.ndarray) -> tuple:
        pts = cls.predict_corners(image_bgr)
        w_top = np.linalg.norm(pts[1] - pts[0])
        w_bot = np.linalg.norm(pts[2] - pts[3])
        target_w = int(max(w_top, w_bot))

        h_left = np.linalg.norm(pts[3] - pts[0])
        h_right = np.linalg.norm(pts[2] - pts[1])
        target_h = int(max(h_left, h_right))

        if target_w < 100 or target_h < 100:
            return image_bgr, pts

        target_pts = np.float32([
            [0, 0],
            [target_w - 1, 0],
            [target_w - 1, target_h - 1],
            [0, target_h - 1]
        ])

        M = cv2.getPerspectiveTransform(pts, target_pts)
        warped = cv2.warpPerspective(image_bgr, M, (target_w, target_h), flags=cv2.INTER_CUBIC)
        return warped, pts


class OcclusionRemovalEngine:
    @staticmethod
    def detect_finger_mask(image: np.ndarray) -> np.ndarray:
        h, w = image.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        border_mask = np.zeros((h, w), dtype=np.uint8)
        border_w = int(w * 0.08)
        border_h = int(h * 0.08)
        border_mask[:border_h, :] = 255
        border_mask[-border_h:, :] = 255
        border_mask[:, :border_w] = 255
        border_mask[:, -border_w:] = 255
        
        ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        skin_ycrcb = cv2.inRange(ycrcb, np.array([0, 133, 77]), np.array([255, 173, 127]))
        skin_hsv = cv2.inRange(hsv, np.array([0, 25, 50]), np.array([30, 220, 255]))
        combined_skin = cv2.bitwise_and(skin_ycrcb, skin_hsv)
        candidate_mask = cv2.bitwise_and(combined_skin, combined_skin, mask=border_mask)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        candidate_mask = cv2.morphologyEx(candidate_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
        contours, _ = cv2.findContours(candidate_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in contours:
            if cv2.contourArea(c) > (w * h * 0.002):
                cv2.drawContours(mask, [c], -1, 255, -1)
        return mask

    @staticmethod
    def inpaint_fingers(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
        if np.count_nonzero(mask) == 0:
            return image
        dilated_mask = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)), iterations=2)
        return cv2.inpaint(image, dilated_mask, inpaintRadius=5, flags=cv2.INPAINT_TELEA)


class IlluminationRegularizationEngine:
    @staticmethod
    def whiten_paper_vflat_style(image: np.ndarray, whiteness_gain: float = 1.15) -> np.ndarray:
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        h, w = l.shape
        scale = 800.0 / max(h, w)
        sw, sh = int(w * scale), int(h * scale)
        l_small = cv2.resize(l, (sw, sh))
        k_size = 51
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k_size, k_size))
        bg_small = cv2.morphologyEx(l_small, cv2.MORPH_CLOSE, kernel)
        bg_small = cv2.GaussianBlur(bg_small, (51, 51), 0)
        bg = cv2.resize(bg_small, (w, h))
        
        l_float = l.astype(np.float32)
        bg_float = np.maximum(bg.astype(np.float32), 1.0)
        l_norm = (l_float / bg_float) * 235.0
        
        l_white = np.clip(l_norm * whiteness_gain, 0, 255)
        paper_mask = l_white > 220
        l_white[paper_mask] = 220 + (l_white[paper_mask] - 220) * 1.0
        l_final = np.clip(l_white, 0, 255).astype(np.uint8)
        
        lab_clean = cv2.merge([l_final, a, b])
        result = cv2.cvtColor(lab_clean, cv2.COLOR_LAB2BGR)
        blur = cv2.GaussianBlur(result, (0, 0), 3.0)
        enhanced = cv2.addWeighted(result, 1.2, blur, -0.2, 0)
        return np.clip(enhanced, 0, 255).astype(np.uint8)

print("AI Model and Vision engines loaded successfully!")

## 📄 Step 4: End-to-End Orchestrator & PDF Compiler
Orchestrates Deep Learning Perspective Warp, Inpainting, and Paper Whitening.

In [ ]:
# @title 📦 End-to-End Pipeline & Multi-Page PDF Compilation
import img2pdf
from PIL import Image
import glob

class ScannerPipelineOrchestrator:
    def __init__(self, enable_ai_warp: bool = True,
                 enable_finger_removal: bool = True, enable_whitening: bool = True):
        self.enable_ai_warp = enable_ai_warp
        self.enable_finger_removal = enable_finger_removal
        self.enable_whitening = enable_whitening
        
    def process_frame(self, image: np.ndarray) -> dict:
        timings = {}
        stages = {'0_raw': image.copy()}
        current = image.copy()
        
        # 1. AI 4-Corner Detection & Perspective Homography
        t0 = time.time()
        if self.enable_ai_warp:
            current, pts = AIDocumentCornerDetector.warp_perspective(current)
            stages['corners'] = pts
        timings['ai_perspective_warp_ms'] = round((time.time() - t0) * 1000, 1)
        stages['1_ai_warped'] = current.copy()
        
        # 2. Finger Removal
        t0 = time.time()
        if self.enable_finger_removal:
            mask = OcclusionRemovalEngine.detect_finger_mask(current)
            stages['finger_mask'] = mask.copy()
            current = OcclusionRemovalEngine.inpaint_fingers(current, mask)
        timings['finger_removal_ms'] = round((time.time() - t0) * 1000, 1)
        stages['2_inpainted'] = current.copy()
        
        # 3. Paper Whitening & Illumination Regularization
        t0 = time.time()
        if self.enable_whitening:
            current = IlluminationRegularizationEngine.whiten_paper_vflat_style(current)
        timings['whitening_ms'] = round((time.time() - t0) * 1000, 1)
        stages['3_whitened_final'] = current.copy()
        
        total_ms = sum(timings.values())
        timings['total_pipeline_ms'] = round(total_ms, 1)
        return {'final': current, 'stages': stages, 'timings': timings}

    @staticmethod
    def compile_batch_to_pdf(processed_image_paths: list, output_pdf_path: str = "scanned_document.pdf") -> str:
        if not processed_image_paths:
            raise ValueError("No processed image files provided for PDF compilation.")
            
        a4_in_pt = (img2pdf.mm_to_pt(210), img2pdf.mm_to_pt(297))
        layout_fun = img2pdf.get_layout_fun(pagesize=a4_in_pt, fit=img2pdf.FitMode.into)
        
        with open(output_pdf_path, "wb") as f:
            f.write(img2pdf.convert(processed_image_paths, layout_fun=layout_fun))
            
        print(f"✅ Successfully created Multi-Page PDF: {output_pdf_path} ({len(processed_image_paths)} pages)")
        return output_pdf_path

print("Orchestrator ready!")

## 📤 Step 5: Test With Your Own Images (Batch Mode ➔ Multi-Page PDF)
Processes all images in `user_test_images/` with the Deep Learning model and creates `My_Scanned_Book.pdf`.

In [ ]:
# @title 🚀 Process Your Images With Deep Learning AI
from google.colab import files
import glob
import os
import matplotlib.pyplot as plt

user_upload_dir = "user_test_images"
os.makedirs(user_upload_dir, exist_ok=True)
os.makedirs("pipeline_outputs/user_processed", exist_ok=True)

image_extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
input_images = []
for ext in image_extensions:
    input_images.extend(glob.glob(os.path.join(user_upload_dir, ext)))
    input_images.extend(glob.glob(os.path.join("input_images", ext)))
input_images = sorted(list(set(input_images)))

if not input_images:
    print("📁 No images found. Click below to upload:")
    uploaded = files.upload()
    if uploaded:
        for fn in uploaded.keys():
            target = os.path.join(user_upload_dir, fn)
            with open(target, 'wb') as f:
                f.write(uploaded[fn])
            input_images.append(target)

if not input_images:
    print("⚠️ No images provided.")
else:
    print(f"\n🚀 Starting Deep Learning processing of {len(input_images)} image(s)...")
    orchestrator = ScannerPipelineOrchestrator()
    processed_output_paths = []
    
    for idx, img_path in enumerate(input_images):
        filename = os.path.basename(img_path)
        print(f"\nProcessing [{idx+1}/{len(input_images)}]: {filename} with Deep Learning...")
        raw_bgr = cv2.imread(img_path)
        if raw_bgr is None:
            print(f"  ❌ Failed to decode {filename}, skipping.")
            continue
            
        res = orchestrator.process_frame(raw_bgr)
        
        out_filename = f"page_{idx+1:03d}_cleaned.jpg"
        out_path = os.path.join("pipeline_outputs/user_processed", out_filename)
        cv2.imwrite(out_path, res['final'])
        processed_output_paths.append(out_path)
        
        # Display Before vs After
        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        axes[0].imshow(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Original: {filename}", fontsize=13)
        axes[0].axis('off')
        
        axes[1].imshow(cv2.cvtColor(res['final'], cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"AI Scanned ({res['timings']['total_pipeline_ms']} ms)", fontsize=13)
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()
        
    if processed_output_paths:
        final_pdf_path = "pipeline_outputs/My_Scanned_Book.pdf"
        ScannerPipelineOrchestrator.compile_batch_to_pdf(processed_output_paths, final_pdf_path)
        print(f"\n🎉 Successfully generated: {final_pdf_path} ({len(processed_output_paths)} pages)")
        files.download(final_pdf_path)


## 🌐 Step 6: Interactive Web Playground (Gradio)
Interactive playground with live single-image testing.

In [ ]:
# @title 🎛️ Launch Interactive Web App (Gradio)
import gradio as gr

def scan_interface(input_image, enable_finger, enable_whitening):
    if input_image is None:
        return None, None, "Please provide an input image."
        
    bgr = cv2.cvtColor(input_image, cv2.COLOR_RGB2BGR)
    orch = ScannerPipelineOrchestrator(
        enable_ai_warp=True,
        enable_finger_removal=enable_finger,
        enable_whitening=enable_whitening
    )
    
    res = orch.process_frame(bgr)
    final_bgr = res['final']
    
    os.makedirs("pipeline_outputs", exist_ok=True)
    page_path = "pipeline_outputs/web_scanned_page.jpg"
    cv2.imwrite(page_path, final_bgr)
    
    pdf_out = "pipeline_outputs/scanned_single.pdf"
    ScannerPipelineOrchestrator.compile_batch_to_pdf([page_path], pdf_out)
    
    final_rgb = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2RGB)
    timing_str = "\n".join([f"{k}: {v} ms" for k, v in res['timings'].items()])
    return final_rgb, pdf_out, timing_str

with gr.Blocks(title="AI Book Scanner Engine") as demo:
    gr.Markdown("## 📖 AI Document & Book Scanner Playground")
    gr.Markdown("Powered by MobileNetV3 Deep Learning 4-Corner Regressor.")
    
    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="numpy", label="Raw Document Photo")
            chk_finger = gr.Checkbox(value=True, label="Finger / Thumb Inpainting")
            chk_white = gr.Checkbox(value=True, label="Paper Whitening & Shadow Removal")
            btn_process = gr.Button("🚀 Process Page With Deep Learning", variant="primary")
            
        with gr.Column():
            output_img = gr.Image(type="numpy", label="AI Perspective Warped & Whitened Output")
            output_pdf = gr.File(label="Download High-Resolution PDF")
            output_timings = gr.Textbox(label="Execution Timing Breakdown", lines=6)
            
    btn_process.click(
        fn=scan_interface,
        inputs=[input_img, chk_finger, chk_white],
        outputs=[output_img, output_pdf, output_timings]
    )

demo.launch(share=True, debug=False)